In [1]:
"""
2026 tariff outlook

Estimates what Canada's September 2026 counter-tariffs could mean for shopper items and food ingredients:
- the yearly tariff bill on US goods (US imports x tariff rate), using 2024 and the tariff-free window
- a what-if for shelf prices, using how much of the 2025 tariff reached shelves (about 12%)
"""
import pandas as pd

pd.set_option("display.max_rows", 100)

tariffs_2026 = pd.read_csv("../data/raw/tariffs/counter_tariffs_2026_09.csv", dtype=str)
us_imports = pd.read_csv("../data/processed/us_imports_by_period.csv", dtype={"hs6_code": str})

print(len(tariffs_2026), "tariff lines on the 2026 list")
print(len(us_imports), "product codes with import data")

629 tariff lines on the 2026 list
5579 product codes with import data


In [2]:
# The 2026 list's food and personal-care items, grouped into the categories shown on the dashboard.
# Each category is a 4-digit HS code; shopper items are bought directly, ingredients go into other products.
categories = {
    "0406": ("Cheese", "shopper item"),
    "0409": ("Honey", "shopper item"),
    "3303": ("Perfume", "shopper item"),
    "3304": ("Make-up", "shopper item"),
    "3305": ("Hair products", "shopper item"),
    "0402": ("Milk powder", "ingredient"),
    "0404": ("Whey", "ingredient"),
    "1703": ("Molasses", "ingredient"),
    "1901": ("Baking mixes", "ingredient"),
}

tariffs_2026["category_code"] = tariffs_2026["tariff_item"].str[:4]
items_2026 = tariffs_2026[tariffs_2026["category_code"].isin(categories)].copy()

items_2026["item"] = items_2026["category_code"].map(lambda code: categories[code][0])
items_2026["type"] = items_2026["category_code"].map(lambda code: categories[code][1])
items_2026["hs6_code"] = items_2026["tariff_item"].str[:7].str.replace(".", "", regex=False)   # "0406.90" -> "040690"
items_2026["rate"] = items_2026["rate"].astype(int) / 100                                       # 25 -> 0.25

# The list has several 8-digit lines per 6-digit code; keep one row per 6-digit code
items_2026 = items_2026[["item", "type", "hs6_code", "rate"]].drop_duplicates()
items_2026

,item,type,hs6_code,rate
0,Milk powder,ingredient,040210,0.50
2,Milk powder,ingredient,040221,0.50
6,Milk powder,ingredient,040291,0.50
8,Milk powder,ingredient,040299,0.50
10,Whey,ingredient,040410,0.50
14,Whey,ingredient,040490,0.50
16,Cheese,shopper item,040610,0.25
18,Cheese,shopper item,040620,0.25
22,Cheese,shopper item,040630,0.25
24,Cheese,shopper item,040640,0.25


In [3]:
# Tariff bill per item = US imports x tariff rate, for 2024 and for the tariff-free window ($ millions)
bills = items_2026.merge(us_imports, on="hs6_code", how="left")

bills["bill_2024"] = bills["us_imports_cad_2024"] * bills["rate"]
bills["bill_tariff_free"] = bills["us_imports_cad_tariff_free"] * bills["rate"]

columns_to_total = ["us_imports_cad_2024", "us_imports_cad_tariff_free", "bill_2024", "bill_tariff_free"]
bill_by_item = bills.groupby(["item", "type", "rate"])[columns_to_total].sum() / 1_000_000

bill_by_item.round(1).sort_values("bill_2024", ascending=False)

,,,us_imports_cad_2024,us_imports_cad_tariff_free,bill_2024,bill_tariff_free
item,type,rate,,,,
Hair products,shopper item,0.50,386.4,379.3,193.2,189.7
Baking mixes,ingredient,0.50,367.6,330.8,183.8,165.4
Make-up,shopper item,0.50,304.1,228.5,152.0,114.3
Perfume,shopper item,0.50,180.6,172.9,90.3,86.5
Whey,ingredient,0.50,106.7,166.4,53.4,83.2
Cheese,shopper item,0.25,168.1,173.9,42.0,43.5
Milk powder,ingredient,0.50,26.2,42.7,13.1,21.3
Molasses,ingredient,0.50,23.5,12.5,11.8,6.2
Honey,shopper item,0.50,6.8,4.5,3.4,2.3


In [5]:
# Compare the two periods: change in US imports per item, and totals for shopper items vs ingredients
bill_by_item["change_in_us_imports"] = (
    bill_by_item["us_imports_cad_tariff_free"] / bill_by_item["us_imports_cad_2024"] - 1
)

totals_by_type = bill_by_item.groupby("type")[["bill_2024", "bill_tariff_free"]].sum()
totals_by_type["change"] = totals_by_type["bill_tariff_free"] / totals_by_type["bill_2024"] - 1

print(bill_by_item["change_in_us_imports"].map("{:+.0%}".format))
print()
totals_by_type["change"] = totals_by_type["change"].map("{:+.0%}".format)
print(totals_by_type.round(1))

item           type          rate
Baking mixes   ingredient    0.50    -10%
Cheese         shopper item  0.25     +3%
Hair products  shopper item  0.50     -2%
Honey          shopper item  0.50    -33%
Make-up        shopper item  0.50    -25%
Milk powder    ingredient    0.50    +63%
Molasses       ingredient    0.50    -47%
Perfume        shopper item  0.50     -4%
Whey           ingredient    0.50    +56%
Name: change_in_us_imports, dtype: str

              bill_2024  bill_tariff_free change
type                                            
ingredient        262.1             276.2    +5%
shopper item      480.9             436.1    -9%


In [6]:
# Save the 2026 tariff bill table to PostgreSQL for the dashboard
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from db import get_engine

outlook = bill_by_item.reset_index()
outlook.to_sql("tariff_2026_outlook", get_engine(), if_exists="replace", index=False)
print(f"Saved {len(outlook)} items to tariff_2026_outlook")

Saved 9 items to tariff_2026_outlook


In [7]:
# Latest Canada-wide price of block cheese, for the cheese panel on the dashboard
query = """
SELECT date, price
FROM prices
WHERE product = 'Block cheese, 500 grams' AND place = 'Canada'
ORDER BY date DESC
LIMIT 1
"""
pd.read_sql(query, get_engine())

,date,price
0,2026-07-01,7.11
